# Read SDTMIG Fragments — the base graph of SDTMIG v3.4 Appendix D

The table "CDISC Variable-naming Fragments" of SDTMIG v3.4 Appendix D, as published in the pinned SDTMIG v3.4 PDF (cdisc.org sign-in). Sign-in content: the graph, the view and Provenance.json are written locally and not published (`.gitignore`).

1. pin check (`Reference_Versions.json`: the PDF, its SHA-256 and the pages the table is on);
2. the table as text (`cfa.read.read_pdf_table`): pdfplumber finds one table per page; the header `Keyword(s)` | `Fragment` once, as the first row; one cell per column in every row; the rows must be consecutive lines of the page text. Cell text unchanged: a cell that lists two fragments, such as "ST, STD", is one cell;
3. one graph for the source, by its CSVW table description, generated from `schema/sdtmig_fragments.yaml` (`cfa.rule.table_to_graph`, CSVW minimal mode);
4. structural SHACL (generated by gen-shacl): must conform;
5. writes `sdtmig-fragments.ttl` (longturtle) and its graph hash;
6. the Excel view `SDTMIG_Fragments.xlsx`, rebuilt from the graph as written; its text hash must equal the source table's text hash;
7. `Provenance.json`.

In [ ]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT))

import pyshacl
from linkml.generators.shaclgen import ShaclGenerator
from rdflib import Graph

from cfa import SCHEMA_DIR
from cfa.graph import graph_sha256, write_longturtle
from cfa.pins import checked_source_file, reference_versions, table_file
from cfa.provenance import flat, provenance, write_provenance
from cfa.read import read_pdf_table, read_xlsx_table
from cfa.rule import (check_module, columns, csvw_metadata, graph_iri, key_columns, load_module, module_source,
                      pct, table_to_graph)
from cfa.view import schema_sheet, text_sha256, view_frame, write_view

MODULE = SCHEMA_DIR / 'sdtmig_fragments.yaml'
OUT = REPO_ROOT / 'base' / 'sdtmig-fragments'

sv = load_module(MODULE)
check_module(sv)
SOURCE = module_source(sv)
TABLES = list(sv.all_classes())
assert len(TABLES) == 1
T = TABLES[0]
print(SOURCE, T)

## The generated contract is current

The CSVW table description and the SHACL shapes in `schema/` (written by `checks/00_Schema.ipynb`) must equal what the module generates now.

In [ ]:
CSVW_PATH = SCHEMA_DIR / 'csvw' / f'{SOURCE}.csv-metadata.json'
SHACL_PATH = SCHEMA_DIR / 'shacl' / f'{SOURCE}.shacl.ttl'
meta = csvw_metadata(sv, T, f'{pct(T)}.csv')
described = json.loads(CSVW_PATH.read_text(encoding='utf-8'))
assert {k: v for k, v in described.items() if k != '@context'} == {k: v for k, v in meta.items() if k != '@context'}, 'CSVW not current: run checks/00_Schema'
assert SHACL_PATH.read_text(encoding='utf-8') == ShaclGenerator(str(MODULE)).serialize(), 'SHACL not current: run checks/00_Schema'
print('contract current')

## 1–3. Pin check, the table as text, the graph

In [ ]:
name = table_file(SOURCE, T)
PAGES = reference_versions()[SOURCE]['files'][name]['pages']
pdf = checked_source_file(SOURCE, name)
table = read_pdf_table(pdf, PAGES, columns(sv, T))
g = table_to_graph(meta, table)
print(f'{T}: pages {PAGES}, rows {len(table)}, columns {len(table.columns)}, triples {len(g)}')

## 4. Structural SHACL

In [ ]:
conforms, _, report = pyshacl.validate(g, shacl_graph=Graph().parse(SHACL_PATH))
assert conforms, report
print('triples', len(g), '| SHACL conforms')

## 5. Write the graph, its hash

In [ ]:
TTL = OUT / f'{SOURCE}.ttl'
write_longturtle(g, TTL, {'frag': f'https://kerfors.github.io/cdisc-for-all/def/{SOURCE}/{pct(T)}#'})
written = Graph().parse(TTL)
GRAPH_HASH = graph_sha256(written)
assert GRAPH_HASH == graph_sha256(g)
print(TTL.name, GRAPH_HASH)

## 6. The Excel view, lossless

In [ ]:
keys = key_columns(sv, T)
view = view_frame(written, sv, T)
SOURCE_HASH = {T: text_sha256(table, keys)}
VIEW_HASH = {T: text_sha256(view, keys)}
assert VIEW_HASH[T] == SOURCE_HASH[T], 'the view differs from the source table'
print('text hash, source = view:', VIEW_HASH[T])

## 7. Provenance; write the view and read it back

In [ ]:
prov = provenance(SOURCE, [pdf], [MODULE, CSVW_PATH, SHACL_PATH], graph_iri(SOURCE),
                  len(written), GRAPH_HASH, SOURCE_HASH, VIEW_HASH)
write_provenance(OUT / 'Provenance.json', prov)
xlsx = OUT / 'SDTMIG_Fragments.xlsx'
readme = [
    f'SDTMIG v3.4 Appendix D, "{T}" ({pdf.name}, pages {PAGES[0]}-{PAGES[-1]}), as published. Sign-in content: not for redistribution.',
    f'Sheet "{T}": rebuilt from the graph {TTL.name}; columns in published order, rows sorted by the key ({", ".join(keys)}).',
    'Every cell is text. The Schema sheet gives each column\'s property IRI; Provenance gives the pins and the hashes.',
    'cdisc-for-all: https://github.com/kerfors/cdisc-for-all',
]
write_view(xlsx, readme, {T: view}, schema_sheet(sv, [T]), flat(prov))
assert text_sha256(read_xlsx_table(xlsx, T), keys) == SOURCE_HASH[T]
print(xlsx.name, 'read back: text hash equal')